# 🎬 ShimiStudio — Cloud Worker v4.5 (רוטציית ענן חינם)

**מנוע וידאו: Wan 2.2 (GGUF Q4)** — פוטוריאליסטי, 5 שנ', לא מצונזר. AnimateDiff+SD1.5 נשאר כמורשת.
הרץ את התאים לפי הסדר. התא האחרון משאיר את ה-worker באוויר — כל עוד הוא רץ, ה-GPU משרת את הסטודיו.

**לפני ההרצה:** הפעל GPU (Kaggle: Settings → Accelerator → GPU T4. Colab: Runtime → Change runtime type → T4 GPU).

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name} | 💾 VRAM: {vram:.1f} GB')
    assert vram >= 10, '❌ צריך לפחות 10GB VRAM'
else:
    raise RuntimeError('❌ אין GPU! הפעל GPU לפי ההוראות למעלה והרץ מחדש')

In [ ]:
import os, subprocess
COMFY_PATH = '/content/ComfyUI'
CUSTOM = COMFY_PATH + '/custom_nodes'
if not os.path.exists(COMFY_PATH):
    print('⬇️ מתקין ComfyUI...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', COMFY_PATH], check=True)
os.chdir(COMFY_PATH)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
nodes = {
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-AnimateDiff-Evolved': 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git',
    'ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    'ComfyUI-ReActor': 'https://github.com/Gourieff/ComfyUI-ReActor.git',
    'ComfyUI-IPAdapter-plus': 'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git',
}
for nm, url in nodes.items():
    p = f'{CUSTOM}/{nm}'
    if not os.path.exists(p):
        print(f'⬇️ {nm}')
        subprocess.run(['git', 'clone', '--depth', '1', url, p], check=True)
subprocess.run(['pip', 'install', '-q', 'imageio-ffmpeg', 'imageio[ffmpeg]'], check=True)
print('✅ ComfyUI + custom nodes מוכנים')

In [ ]:
import os, requests, shutil
M = '/content/ComfyUI/models'
# ── Google Drive: מטמון מודלים קבוע — הורדה חד-פעמית, הרצות הבאות מהירות ──
DRIVE_DIR = '/content/drive/MyDrive/ShimiStudio/models'
USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    USE_DRIVE = True
    print('✅ Drive מחובר — מודלים יישמרו ב-Drive ולא יורדו שוב')
except Exception as e:
    print(f'⚠️ Drive לא מחובר ({e}) — ממשיכים בלי מטמון קבוע')
downloads = [
    (M + '/diffusion_models', 'Wan2.2-TI2V-5B-Q4_K_M.gguf', 'https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-Q4_K_M.gguf'),
    (M + '/text_encoders', 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
    (M + '/vae', 'Wan2.2_VAE.safetensors', 'https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/VAE/Wan2.2_VAE.safetensors'),
    (M + '/checkpoints', 'DreamShaper_8_pruned.safetensors', 'https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors'),
    (M + '/checkpoints', 'CyberRealistic_V1.1.safetensors', 'https://huggingface.co/cyberdelia/CyberRealistic/resolve/main/CyberRealistic_V1.1.safetensors'),
    (M + '/animatediff_models', 'mm_sd_v15_v2.ckpt', 'https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt'),
    (M + '/ipadapter', 'ip-adapter-plus_sd15.safetensors', 'https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sd15.safetensors'),
    (M + '/clip_vision', 'CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors', 'https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors'),
]
def fetch(url, dest, chunk=1024*1024):
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    done = 0
    with open(dest, 'wb') as f:
        for c in r.iter_content(chunk_size=chunk):
            f.write(c)
            done += len(c)
            if total and (done*100//total) % 10 == 0:
                print(f'   {done*100//total}% ({done//1048576}/{total//1048576} MB)', flush=True)
for d, fname, url in downloads:
    os.makedirs(d, exist_ok=True)
    dest = os.path.join(d, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 1e6:
        print(f'✅ {fname} כבר קיים')
        continue
    ddest = os.path.join(DRIVE_DIR, fname)
    if USE_DRIVE and os.path.exists(ddest) and os.path.getsize(ddest) > 1e6:
        os.symlink(ddest, dest)
        print(f'✅ {fname} מ-Drive (מטמון)')
        continue
    print(f'⬇️ מוריד {fname}...')
    fetch(url, dest)
    if USE_DRIVE:
        shutil.copyfile(dest, os.path.join(DRIVE_DIR, fname))
        print(f'   💾 נשמר גם ל-Drive')
print('✅ כל המודלים מוכנים')

# ── אופציונלי: LoRA דמות מ-CivitAI (Wan 2.2) — מילאו ואז הריצו שוב ──
CIVITAI_TOKEN = ''        # מפתח API אישי מ-CivitAI
LORA_CIVITAI_ID = None    # מזהה המודל, למשל 1234567
if LORA_CIVITAI_ID and CIVITAI_TOKEN:
    ldir = M + '/loras'
    os.makedirs(ldir, exist_ok=True)
    lname = f'civitai_{LORA_CIVITAI_ID}.safetensors'
    ldest = os.path.join(ldir, lname)
    if not (os.path.exists(ldest) and os.path.getsize(ldest) > 1e5):
        print(f'⬇️ מוריד LoRA {LORA_CIVITAI_ID} מ-CivitAI...')
        r = requests.get(f'https://civitai.com/api/download/models/{LORA_CIVITAI_ID}',
                        headers={'Authorization': f'Bearer {CIVITAI_TOKEN}'},
                        stream=True, timeout=300, allow_redirects=True)
        r.raise_for_status()
        with open(ldest, 'wb') as f:
            for c in r.iter_content(chunk_size=1024*1024):
                f.write(c)
        print('✅ LoRA הורד')


In [ ]:
import os, re, json, secrets, sys, requests
from datetime import datetime, timezone, timedelta
WORKER_DIR = '/content/ShimiWorker'
os.makedirs(WORKER_DIR, exist_ok=True)
TOKEN = secrets.token_hex(12)
NAME = 'Worker-Colab-T4'
API_BASE = 'https://shimi-studio.base44.app/api/apps/6a9206e9f29b8d9f70a77b47'
SESSION_HOURS = 11
src = requests.get('https://raw.githubusercontent.com/sunraz/ShimiStudio/main/worker_v45.py', timeout=60).text
assert 'workerApi' in src and 'WanImageToVideo' in src, 'קובץ ה-worker v4.5 לא ירד תקין'
src = re.sub(r'VENV_PY = .*', 'VENV_PY = sys.executable', src, count=1)
exp = (datetime.now(timezone.utc) + timedelta(hours=SESSION_HOURS)).isoformat()
src = src.replace('"action":"register",', '"action":"register","worker_type":"cloud","session_expires_at":"' + exp + '","engines":["wan22","ad15"],', 1)
with open(WORKER_DIR + '/worker.py', 'w') as f:
    f.write(src)
CONFIG = {"server": API_BASE.rsplit('/api/apps', 1)[0], "token": TOKEN, "name": NAME,
    "apiBase": API_BASE, "comfyui_path": '/content/ComfyUI',
    "comfyui_url": "http://127.0.0.1:8188", "default_engine": "wan22",
    "chain_shots": True, "output_size": [1080, 1600], "output_fps": 30, "interp_mode": "mci",
    "wan": {"model": "Wan2.2-TI2V-5B-Q4_K_M.gguf", "text_encoder": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
             "vae": "Wan2.2_VAE.safetensors", "lora": None, "lora_strength": 0.8}}
with open(WORKER_DIR + '/config.json', 'w') as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)
print(f'✅ Worker v4.5 (cloud) מוכן | שם: {NAME} | תוקף סשן: {SESSION_HOURS} שעות')
print(f'   מנוע: Wan 2.2 GGUF | שרשור שוטים (I2V) | גימור: 1080x1600 @30fps')
print(f'   Token: {TOKEN[:8]}...')

In [ ]:
import os, sys
os.chdir('/content/ShimiWorker')
print('🚀 מעלה את ComfyUI ומתחבר לתור של ShimiStudio...')
print('   השאר את התא הזה רץ — כל עוד הוא רץ, ה-GPU הזה משרת את הסטודיו')
print()
raise SystemExit(os.system(sys.executable + ' worker.py') >> 8)